# DeBERTa smoke test — increment 2

This runs a bounded, non-scientific smoke test through the authors' unmodified `train_head.py` and DeepSpeed path. It uses 16 SQuAD contexts, expects four optimizer steps, and stores the checkpoint outside Git.

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_URL = "https://github.com/apusharma45/indirect-pia-detection.git"
PROJECT_DIR = Path("/kaggle/working/indirect-pia-detection")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f"Refusing to clone into non-empty directory: {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

## Run the smoke test

The runner records dependency-installation or training failures instead of fabricating results. A nonzero return code is evidence to inspect; the notebook intentionally continues to the review and push cells.

In [ ]:
smoke = subprocess.run(["bash", "scripts/run_deberta_smoke.sh", "--install-deps"])
print("Smoke runner return code:", smoke.returncode)

## Review the structured result

Success here means only that the pipeline ran and saved a checkpoint. It is not a paper-comparable result.

In [ ]:
RESULT_DIR = PROJECT_DIR / "results/deberta_smoke"
for name in ["metrics.json", "error.txt", "environment.txt", "command.txt"]:
    path = RESULT_DIR / name
    if path.exists():
        print(f"\n===== {name} =====")
        print(path.read_text(encoding="utf-8", errors="replace"))

log_path = RESULT_DIR / "training_log.txt"
if log_path.exists():
    print("\n===== final training log lines =====")
    print("".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-80:]))

## Commit the small evidence files

Create a Kaggle secret named `GITHUB_TOKEN` containing a GitHub token with Contents read/write access to this repository. This cell stages only `results/deberta_smoke/`; checkpoints under `artifacts/models/` remain ignored.

In [ ]:
import base64
from kaggle_secrets import UserSecretsClient

GITHUB_USER = "apusharma45"
GITHUB_EMAIL = "129539180+apusharma45@users.noreply.github.com"
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Kaggle secret GITHUB_TOKEN is missing.")
encoded = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()
auth_option = f"http.extraHeader=AUTHORIZATION: basic {encoded}"

def authenticated_git(*args):
    result = subprocess.run(["git", "-c", auth_option, *args], cwd=PROJECT_DIR)
    if result.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed with exit code {result.returncode}")

subprocess.run(["git", "config", "user.name", GITHUB_USER], cwd=PROJECT_DIR, check=True)
subprocess.run(["git", "config", "user.email", GITHUB_EMAIL], cwd=PROJECT_DIR, check=True)
subprocess.run(["git", "add", "--", "results/deberta_smoke"], cwd=PROJECT_DIR, check=True)
staged = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=PROJECT_DIR)
if staged.returncode == 1:
    subprocess.run(
        ["git", "commit", "-m", "test: record kaggle deberta smoke result"],
        cwd=PROJECT_DIR,
        check=True,
    )
elif staged.returncode != 0:
    raise RuntimeError(f"git diff failed with exit code {staged.returncode}")
authenticated_git("pull", "--rebase", "origin", "main")
authenticated_git("push", "origin", "main")
del token, encoded, auth_option
print("Smoke evidence pushed to GitHub.")